Подключение необходимых библиотек:

In [1]:
!pip install python-docx
!pip install -U transformers
import re
import pandas as pd
from docx import Document
import numpy as np
from datasets import Dataset


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 51.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
import torch
torch.cuda.empty_cache()

Чтение транскрипта и кодировок всех файлов для создания общей таблицы

In [3]:
def parse_pul_intervyu(filename):
    doc = Document(filename)
    paragraphs = [para.text.strip() for para in doc.paragraphs if para.text.strip()]
    if not paragraphs:
        return []

    global_topic = ""
    if not re.match(r'^Интервью\s+\d+', paragraphs[0]):
        global_topic = paragraphs[0]

    interviews = []
    current_interview = None
    in_transcript = False

    for text in paragraphs:
        if text == global_topic:
            continue
        match = re.match(r'^Интервью\s+(\d+)', text)
        if match:
            if current_interview:
                interviews.append(current_interview)
            interview_num = match.group(1)
            current_interview = {'interview_num': interview_num, 'topic': global_topic, 'transcript': ''}
            in_transcript = True
        elif current_interview and in_transcript:
            if current_interview['transcript']:
                current_interview['transcript'] += '\n' + text
            else:
                current_interview['transcript'] = text
    if current_interview:
        interviews.append(current_interview)
    return interviews

def parse_kodirovki(filename):
    doc = Document(filename)
    paragraphs = [para.text.strip() for para in doc.paragraphs if para.text.strip()]
    interviews = []
    current_interview = None
    for text in paragraphs:
        match = re.match(r'^Интервью\s+(\d+)', text)
        if match:
            if current_interview:
                interviews.append(current_interview)
            interview_num = match.group(1)
            current_interview = {'interview_num': interview_num, 'coding_text': ''}
        elif current_interview:
            if current_interview['coding_text']:
                current_interview['coding_text'] += '\n' + text
            else:
                current_interview['coding_text'] = text
    if current_interview:
        interviews.append(current_interview)
    return interviews

files = [
    ('pul_intervyu_1.docx', 'kodirovki_pfi_23.docx', 'result1.xlsx'),
    ('pul_intervyu_2.docx', 'kodirovki_pfi_24.docx', 'result2.xlsx'),
    ('pul_intervyu_3.docx', 'kodirovki_sp_24.docx', 'result3.xlsx'),
    ('pul_intervyu_4.docx', 'kodirovki_pfi_25.docx', 'result4.xlsx')
]

all_dfs = []
for trans_file, code_file, out_file in files:
    data1 = parse_pul_intervyu(trans_file)
    data2 = parse_kodirovki(code_file)
    df1 = pd.DataFrame(data1)
    df2 = pd.DataFrame(data2)
    merged = pd.merge(df1, df2, on='interview_num', how='outer')
    merged['sort_key'] = merged['interview_num'].apply(lambda x: int(x) if str(x).isdigit() else 0)
    merged = merged.sort_values('sort_key').drop('sort_key', axis=1).fillna('')

    result = merged[['topic', 'interview_num', 'transcript', 'coding_text']].copy()
    result.insert(0, 'Индекс', range(len(result)))
    result.columns = ['Индекс','Тема интервью', 'Порядковый номер интервью', 'Транскрипт интервью', 'Кодирование интервью']
    result.to_excel(out_file, index=False)
    all_dfs.append(result)
    print(result.head())

full_dataset = pd.concat(all_dfs, ignore_index=True)
full_dataset.insert(0, 'Общий номер', range(len(full_dataset)))
full_dataset.to_excel('full_interviews_dataset.xlsx', index=False)
print(f"Общий датасет: {len(full_dataset)} интервью")

    Индекс                                      Тема интервью  \
0        0  Тема: ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫ...   
11       1  Тема: ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫ...   
22       2  Тема: ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫ...   
23       3  Тема: ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫ...   
24       4  Тема: ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫ...   

   Порядковый номер интервью  \
0                          1   
11                         2   
22                         3   
23                         4   
24                         5   

                                  Транскрипт интервью  \
0   Интервьюер: Так, все, все записи начались. Сна...   
11  Интервьюер: Смотри, вначале расскажи, пожалуйс...   
22  Интервьюер: Так, тогда начинаем. Расскажи спер...   
23  Интервьюер: Тогда, наверное, начнем. Сперва мо...   
24  Интервьюер: Для начала расскажи о себе. Скольк...   

                                 Кодирование интервью  
0   **Общий код 1

In [ ]:
full_dataset

,Общий номер,Индекс,Тема интервью,Порядковый номер интервью,Транскрипт интервью,Кодирование интервью
0,0,0,"Тема: ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫ...",1,"Интервьюер: Так, все, все записи начались. Сна...","**Общий код 1: Поколенческие характеристики, ц..."
1,1,1,"Тема: ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫ...",2,"Интервьюер: Смотри, вначале расскажи, пожалуйс...","**Общий код 1: Поколенческие характеристики, ц..."
2,2,2,"Тема: ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫ...",3,"Интервьюер: Так, тогда начинаем. Расскажи спер...","**Общий код 1: Поколенческие характеристики, ц..."
3,3,3,"Тема: ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫ...",4,"Интервьюер: Тогда, наверное, начнем. Сперва мо...","**Общий код 1: Поколенческие характеристики, ц..."
4,4,4,"Тема: ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫ...",5,Интервьюер: Для начала расскажи о себе. Скольк...,"**Общий код 1: Поколенческие характеристики, ц..."
...,...,...,...,...,...,...
145,145,26,"Тема: МНОГООБРАЗИЕ ФОРМ, ЭФФЕКТОВ И БАРЬЕРОВ И...",27,Интервьюер: Ещё один вопрос. Прежде чем мы нач...,**Общий код 1: Переход к самостоятельности: от...
146,146,27,"Тема: МНОГООБРАЗИЕ ФОРМ, ЭФФЕКТОВ И БАРЬЕРОВ И...",28,Интервьюер: Я из лаборатории молодежных исслед...,**Общий код 1: Жизненный путь и траектории взр...
147,147,28,"Тема: МНОГООБРАЗИЕ ФОРМ, ЭФФЕКТОВ И БАРЬЕРОВ И...",29,"Интервьюер: Да, все, с твоего согласия, я запи...",**Общий код 1: Жизненные траектории и ключевые...
148,148,29,"Тема: МНОГООБРАЗИЕ ФОРМ, ЭФФЕКТОВ И БАРЬЕРОВ И...",30,"Интервьюер: Можете, пожалуйста, рассказать нем...",**Общий код 1: Траектории взросления и ключевы...


Подготовка обучающего и тестового датасета - извлечение кодировок для задачи генерации кодов

In [4]:
def extract_general_codes(coding_text):
    if pd.isna(coding_text) or not coding_text.strip():
        return []

    pattern = r'\*\*Общий код \d+:[^**]+\*\*'
    general_codes = re.findall(pattern, coding_text)
    general_codes = [code[2:-2].strip() for code in general_codes]
    return general_codes

def prepare_training_data(full_df):
    training_data = []

    for _, row in full_df.iterrows():
        if pd.notna(row['Кодирование интервью']) and row['Кодирование интервью'].strip():

            # Задача 1: Генерация кодов
            training_data.append({
                'id': row['Общий номер'],
                'task': 'code_generation',
                'input_text': f"{row['Тема интервью']}\n\nТранскрипт: {row['Транскрипт интервью']}",
                'target': row['Кодирование интервью']
            })

            # Задача 2: Разметка с общими кодами
            general_codes = extract_general_codes(row['Кодирование интервью'])
            if general_codes:
                codes_list = "; ".join(general_codes)
                training_data.append({
                    'id': row['Общий номер'],
                    'task': 'text_markup_with_codes',
                    'input_text': f"Тема: {row['Тема интервью']}\nТранскрипт: {row['Транскрипт интервью']}\nОбщие коды: {codes_list}",
                    'target': row['Кодирование интервью']
                })

    return pd.DataFrame(training_data)

data = prepare_training_data(full_dataset)

In [ ]:
data.head()

,id,task,input_text,target
0,0,code_generation,"Тема: ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫ...","**Общий код 1: Поколенческие характеристики, ц..."
1,0,text_markup_with_codes,"Тема: Тема: ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕН...","**Общий код 1: Поколенческие характеристики, ц..."
2,1,code_generation,"Тема: ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫ...","**Общий код 1: Поколенческие характеристики, ц..."
3,1,text_markup_with_codes,"Тема: Тема: ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕН...","**Общий код 1: Поколенческие характеристики, ц..."
4,2,code_generation,"Тема: ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫ...","**Общий код 1: Поколенческие характеристики, ц..."


Разделение на test и train

In [5]:
def split_dataset_properly(train_df):
    if 'id' not in train_df.columns:
        return None, None, None

    interview_ids = train_df['id'].unique()
    np.random.shuffle(interview_ids)

    n_train = int(0.7 * len(interview_ids))
    n_val = int(0.15 * len(interview_ids))

    train_ids = interview_ids[:n_train]
    val_ids = interview_ids[n_train:n_train+n_val]
    test_ids = interview_ids[n_train+n_val:]

    train_df_split = train_df[train_df['id'].isin(train_ids)].copy()
    val_df_split = train_df[train_df['id'].isin(val_ids)].copy()
    test_df_split = train_df[train_df['id'].isin(test_ids)].copy()

    print(f"Train: {len(train_df_split)} примеров")
    print(f"Val:   {len(val_df_split)} примеров")
    print(f"Test:  {len(test_df_split)} примеров")

    return train_df_split, val_df_split, test_df_split

train_df, val_df, test_df = split_dataset_properly(data)

train_df.to_pickle('train_split.pkl')
val_df.to_pickle('val_split.pkl')
test_df.to_pickle('test_split.pkl')

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

Train: 210 примеров
Val:   44 примеров
Test:  46 примеров


Мы прибегаем к стратифицированному разделению по уникальным ID интервью (70/15/15). Таким образом, каждому сплиту достаются данные только из своих ID, что защищает модель от "запоминания интервью", позволяя лишь обобщать на новые.

Дообучение модели

In [6]:
!pip install -q transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.3 MB/s eta 0:00:00


In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, BitsAndBytesConfig, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model

Запуск и работа с моделью - QWEN

In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model_name = "Qwen/Qwen3.5-2B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Формирование промтов (instruction tuning)


In [9]:
def build_prompt(example):
    if example['task'] == 'code_generation':
        instruction = (
            "Сгенерируй тематические коды и разметь интервью. "
            "Используй формат: <код>цитата</код>."
        )
    else:
        instruction = (
            "Разметь интервью с использованием данных кодов. "
            "Используй формат: <код>цитата</код>."
        )

    return f"""<|instruction|>
{instruction}

<|input|>
Интервью:
{example['input_text']}

<|output|>
{example['target']}{tokenizer.eos_token}"""

In [10]:
def format_dataset(dataset):
    return dataset.map(lambda x: {"text": build_prompt(x)})

train_dataset = format_dataset(train_dataset)
val_dataset = format_dataset(val_dataset)

Map:   0%|          | 0/210 [00:00<?, ? examples/s]

Map:   0%|          | 0/44 [00:00<?, ? examples/s]

Токенизация



In [11]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=256,
        padding=False
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens


train_dataset = train_dataset.map(tokenize, batched=True)

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)


data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

Map:   0%|          | 0/210 [00:00<?, ? examples/s]

LoRA

In [12]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 417,792 || all params: 1,882,242,880 || trainable%: 0.0222


Training

In [13]:
training_args = TrainingArguments(
    output_dir="./results",

    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,

    num_train_epochs=3,
    learning_rate=2e-4,

    logging_steps=5,

    save_total_limit=2,
    bf16=True,

    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)


In [14]:
import gc
gc.collect()

import torch
torch.cuda.empty_cache()

In [15]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
5,2.644479
10,2.349692
15,2.565252
20,2.305111
25,2.307301
30,2.101411
35,2.030940
40,1.836771
45,1.548065
50,1.675724


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


TrainOutput(global_step=630, training_loss=1.3566235565003895, metrics={'train_runtime': 827.0454, 'train_samples_per_second': 0.762, 'train_steps_per_second': 0.762, 'total_flos': 1329286068633600.0, 'train_loss': 1.3566235565003895, 'epoch': 3.0})

In [16]:
model.save_pretrained("./qwen_lora")
tokenizer.save_pretrained("./qwen_lora")
model.config.use_cache = False
model.gradient_checkpointing_enable()

('./qwen_lora/tokenizer_config.json',
 './qwen_lora/chat_template.jinja',
 './qwen_lora/tokenizer.json')

Функция генерации

In [30]:
def generate_codes(example, max_new_tokens=128):
    prompt = build_prompt(example)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id
        )

    generated_tokens = outputs[0][input_length:]

    result = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return result.strip()

In [31]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

In [ ]:
predictions = []

for example in test_dataset:
    pred = generate_codes(example)

    predictions.append({
        "id": example["id"],
        "task": example["task"],
        "prediction": pred,
        "target": example["target"]
    })

pred_df = pd.DataFrame(predictions)
pred_df.head()

In [28]:
pred_df['prediction'][2]

'**Общий код 1: Поколенческие характеристики, ценности и жизненные ориентиры**\n"Мое поколение - люди, рожденные там 90-91 год? [...] люди, рожденные с 90-го года. 91-й, то есть перестройка, потому что мой муж, он другого поколения, все-таки, он 84 года рождения, мой брат 85, это люди, видевшие Советский Союз, видевшие другую жизнь с ее плюсами и минусами. Мне кажется, мои друзья, подруги. Мне кажется, мы такие более... Прагматичные, что ли, рассчитываем на себя." - **Прагматичность и опора на себя как черта поколения (конкретный код)**\n"Мы видели сложности жизни в девяностые. У всех семья были разные, да, но в целом мы помним, что было тяжело. А потом... Та школа, в которой я росла, мой садик, в нас как-то поддерживали свободолюбие и все такое, поэтому, не знаю, может, поэтому мне сложно ужиться в компании и работать на кого-то в строгом графике. Я очень ценю свою свободу, творческую в том числе." - **Свободолюбие и нежелание работать в строгом графике (конкретный код)**\n"Те, кому п